# 🗂️ Notebook 2: Discord — Data Model & APIs


## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

All code in this lab is **self-contained Python** — no servers, no databases. We simulate Discord's gateway, pub/sub, and fan-out in memory so you can run every cell and see what happens.


## 💡 What we'll build

In this notebook we design the **data shapes** (tables, IDs, events) and the **APIs** a Discord-like client would use. We'll write runnable code for:

1. **Snowflake IDs** — Discord's time-ordered 64-bit message IDs.
2. **Pagination**: offset (bad) vs cursor (best).
3. **Gateway events** as Pydantic models.
4. **Heartbeats** — how the server detects zombie sockets.
5. **Reconnect & resume** using a sequence number.
6. **Idempotency** with a client nonce so retries don't duplicate.


## 🧾 Core entities

| Table | Columns | Notes |
|---|---|---|
| `users` | `id`, `name`, `avatar`, `presence` | |
| `guilds` | `id`, `name`, `owner_id` | a "server" in Discord |
| `channels` | `id`, `guild_id`, `type`, `name` | `type` ∈ {text, voice} |
| `memberships` | `user_id`, `guild_id`, `roles[]` | join table |
| `messages` | `id`, `channel_id`, `author_id`, `content`, `ts` | **partition by `channel_id`** |

### Why partition messages by `channel_id`?

99% of reads are *"give me the last 50 messages in this channel"*. If all messages are stored together, that query has to scan or index across the whole world. If they are **partitioned by channel**, the database only has to look at one shard. Cassandra and ScyllaDB are the usual picks because they natively support *wide-column, per-partition ordering*.


## ❄️ Snowflake IDs — time-ordered 64-bit IDs

Discord uses Snowflake IDs (invented at Twitter). A Snowflake packs three things into a 64-bit integer:

```
│   42 bits: millis since epoch   │ 10 bits: worker id │ 12 bits: seq │
```

**Why this is clever:**
- IDs are **roughly time-ordered** — sorting by id = sorting by creation time, no extra index needed.
- No central counter → each worker generates IDs independently (horizontal scaling).
- 4096 sequence bits per ms per worker = plenty of headroom.


In [ ]:
# ==== Snowflake ID generator ====
import time, threading

EPOCH_MS = 1_420_070_400_000  # 2015-01-01, like Discord's real epoch

class Snowflake:
    def __init__(self, worker_id: int):
        assert 0 <= worker_id < 1024, 'worker_id must fit in 10 bits'
        self.worker_id = worker_id
        self.seq = 0
        self.last_ms = -1
        self.lock = threading.Lock()

    def next_id(self) -> int:
        with self.lock:
            now = int(time.time() * 1000)
            if now == self.last_ms:
                self.seq = (self.seq + 1) & 0xFFF  # 12-bit wrap
                if self.seq == 0:
                    # we burned through 4096 IDs in this ms — wait for the next one
                    while now <= self.last_ms:
                        now = int(time.time() * 1000)
            else:
                self.seq = 0
            self.last_ms = now
            return ((now - EPOCH_MS) << 22) | (self.worker_id << 12) | self.seq

    @staticmethod
    def decode(sf: int):
        ts_ms   = (sf >> 22) + EPOCH_MS
        worker  = (sf >> 12) & 0x3FF
        seq     = sf & 0xFFF
        return {'ts_ms': ts_ms, 'worker': worker, 'seq': seq}

gen = Snowflake(worker_id=17)
ids = [gen.next_id() for _ in range(5)]
for i in ids:
    print(i, Snowflake.decode(i))

# Crucially, sorting by id == sorting by time — free pagination order!
assert ids == sorted(ids)
print('\nIDs are strictly increasing ✓')


## 📜 Pagination: bad → best

Loading scrollback is just *"give me messages before this point"*. We'll show why `LIMIT/OFFSET` is a trap, and why a **cursor over Snowflake IDs** is the fix.


In [ ]:
# ==== Pagination comparison ====
import bisect, time

# Imagine messages stored in sorted order by id (Snowflake ⇒ sorted by time).
N = 200_000
messages = [(i, f'msg-{i}') for i in range(N)]
ids = [m[0] for m in messages]
PAGE = 50

# ---------- ❌ BAD: OFFSET pagination ----------
# SQL does `LIMIT 50 OFFSET k`. Even with an index, the DB must walk past k rows
# to find the starting point. We mimic that with a real loop (no Python slicing trick).
def offset_page(offset):
    i = 0
    for _ in messages:         # DB walking the index
        if i == offset:
            break
        i += 1
    return messages[offset: offset + PAGE]

# ---------- ✅ BEST: cursor on id ----------
# 'Give me 50 messages with id < cursor' — O(log N) via the index, independent of page depth.
def cursor_page(before_id):
    j = bisect.bisect_left(ids, before_id)
    start = max(0, j - PAGE)
    return messages[start:j]

# Scroll to page 100, 500, 1000, 2000 (deeper pages).
depths = [100, 500, 1000, 2000]
for d in depths:
    t0 = time.perf_counter()
    offset_page(d * PAGE)
    t_off = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    cursor_page(ids[N - d*PAGE])
    t_cur = (time.perf_counter() - t0) * 1000
    print(f'page #{d:>4}  OFFSET {t_off:6.2f} ms   CURSOR {t_cur:6.3f} ms')

print('\nWhy cursor wins:')
print(' - No row counting: the DB jumps straight to the index entry.')
print(' - Stable: new messages arriving don\'t shift page boundaries.')


## 🛰️ Gateway protocol (events)

The WebSocket is the only connection a Discord client needs for real-time. Both directions send JSON frames with an **opcode**.

```http
C→S:  { "op": "identify", "d": { "token": "..." } }
S→C:  { "op": "hello",    "d": { "heartbeat_ms": 30000 } }
S→C:  { "op": "ready",    "d": { "user": {...}, "guilds": [...] } }
S→C:  { "op": "MESSAGE_CREATE",   "s": 42, "d": { "channel_id":..., "content":... } }
S→C:  { "op": "PRESENCE_UPDATE",  "s": 43, "d": { "user_id":..., "status":"online" } }
C→S:  { "op": "heartbeat",        "d": 43 }   # last s we saw
C→S:  { "op": "resume",           "d": { "session": "...", "seq": 43 } }
```

The field `s` is a **monotonically increasing sequence number per session**. After a disconnect, the client sends `resume` with the last `s` it received and the server replays anything newer. We'll simulate that below.


In [ ]:
# ==== Typed gateway events with pydantic ====
from pydantic import BaseModel, Field
from datetime import datetime, timezone
from typing import Literal, Optional

class Message(BaseModel):
    id: int
    channel_id: int
    author_id: int
    content: str
    ts: datetime
    nonce: Optional[str] = None   # set by client to dedupe retries

class GatewayEvent(BaseModel):
    op: Literal['hello','ready','MESSAGE_CREATE','PRESENCE_UPDATE','heartbeat','heartbeat_ack','resume','resumed']
    s: Optional[int] = None       # sequence number (server→client events)
    d: Optional[dict] = None      # data payload

evt = GatewayEvent(
    op='MESSAGE_CREATE', s=42,
    d=Message(id=gen.next_id(), channel_id=99, author_id=42,
              content='hi', ts=datetime.now(timezone.utc), nonce='abc').model_dump(mode='json'),
)
print(evt.model_dump_json(indent=2))


## ❤️ Heartbeats — detecting dead sockets

TCP can keep a connection *"open"* long after the network died (laptop lid closed, wifi dropped, phone tunneled into a subway). Without an app-level heartbeat, the server would keep fanning messages into a black hole.

**Pattern:** every ~30 s the client sends `heartbeat`; the server replies `heartbeat_ack`. Miss 2 in a row → assume dead, close the socket, free the resources, let the client reconnect.


In [ ]:
# ==== Heartbeat simulation (synchronous time — no asyncio) ====
class GatewaySession:
    HEARTBEAT_MS = 30_000
    MISSES_ALLOWED = 2

    def __init__(self):
        self.last_heartbeat_at_ms = 0
        self.alive = True

    def on_heartbeat(self, now_ms):
        self.last_heartbeat_at_ms = now_ms
        return {'op': 'heartbeat_ack'}

    def tick(self, now_ms):
        # called by the server's reaper every HEARTBEAT_MS
        if now_ms - self.last_heartbeat_at_ms > self.HEARTBEAT_MS * self.MISSES_ALLOWED:
            self.alive = False

s = GatewaySession()
t = 0
for step in range(5):
    t += 30_000
    s.on_heartbeat(t)          # client sends heartbeat on time
    s.tick(t)
    print(f't={t/1000:>4.0f}s  alive={s.alive}  (healthy)')

# Simulate dropout: no more heartbeats for 90s
for step in range(3):
    t += 30_000
    s.tick(t)
    print(f't={t/1000:>4.0f}s  alive={s.alive}  (silent)')

assert not s.alive
print('\nSession correctly marked dead after 2 missed heartbeats ✓')


## 🔁 Reconnect & resume

If the session is still recent (say, <3 min old), a reconnecting client can `resume` instead of starting over. The server replays every event with `s` greater than what the client saw.

Without `resume`, every metro-tunnel wifi hiccup would cost a full `ready` payload (thousands of guilds). That's why `resume` exists.


In [ ]:
# ==== Resume after a dropped connection ====
from collections import deque

class SessionBuffer:
    """Server-side per-session ring buffer of recent events."""
    def __init__(self, capacity=1000):
        self.buf: deque = deque(maxlen=capacity)
        self.seq = 0

    def publish(self, op, d):
        self.seq += 1
        evt = {'op': op, 's': self.seq, 'd': d}
        self.buf.append(evt)
        return evt

    def resume(self, last_seen_seq):
        # Replay events newer than what the client already has.
        return [e for e in self.buf if e['s'] > last_seen_seq]

sess = SessionBuffer()
for msg in ['hi', 'hello', 'how are you', 'ping', 'pong']:
    sess.publish('MESSAGE_CREATE', {'content': msg})

# Client saw up to s=2, then disconnected. On reconnect it calls resume(2):
missed = sess.resume(last_seen_seq=2)
for e in missed:
    print(e)
print(f'\nReplayed {len(missed)} missed events; no full re-sync needed.')


## 🧪 Idempotency with a client nonce

Mobile networks love to time out *after* the request reached the server. The client retries; now there are two copies of "lol" in the channel. Fix: the **client** attaches a random `nonce` to every send. The server remembers nonces for a few minutes and rejects duplicates.


In [ ]:
# ==== Deduplication by nonce ====
import uuid

class MessageService:
    def __init__(self):
        self.seen_nonces = {}    # nonce -> message_id
        self.messages = []

    def send(self, channel_id, author_id, content, nonce):
        if nonce in self.seen_nonces:
            return {'status': 'duplicate', 'id': self.seen_nonces[nonce]}
        mid = gen.next_id()
        self.seen_nonces[nonce] = mid
        self.messages.append({'id': mid, 'channel_id': channel_id,
                              'author_id': author_id, 'content': content})
        return {'status': 'created', 'id': mid}

svc = MessageService()
n = str(uuid.uuid4())
print(svc.send(1, 42, 'hello', nonce=n))   # first try — created
print(svc.send(1, 42, 'hello', nonce=n))   # retry — duplicate, same id
print(f'Stored messages: {len(svc.messages)} (should be 1)')


## ✅ Summary

- **Snowflake IDs** give us global, time-ordered IDs without a central counter.
- Paginate with **cursors**, never with OFFSET — OFFSET costs more on deeper pages.
- Every gateway event has a **sequence number** so clients can `resume` after a blip.
- **Heartbeats** detect dead sockets that TCP can't see.
- **Nonces** make message send idempotent — retries don't duplicate.

Next: [Notebook 3 — Deep Dive: Fan-out, Presence, Voice](./03_deep_dive.ipynb).
